In [ ]:
import os
import sys

# Add the project root to sys.path
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
# sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
os.chdir("..") # Change working directory to project root

from src.api.core.config import config
from src.api.rag.retrieval import rag_pipeline

import asyncio
from langsmith import Client
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall, NonLLMContextRecall


/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/run_helpers.py:480: UserWarning: Unrecognized run_type: reranker. Must be one of: {'chain', 'retriever', 'embedding', 'prompt', 'llm', 'tool', 'parser'}. Did you mean @traceable(name='reranker')?
  warnings.warn(
/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["EVALUATION_MODE"] = "true"

In [3]:
# Initialize LangSmith & Qdrant clients

ls_client = Client(api_key=config.LANGSMITH_API_KEY)

# qdrant_client = QdrantClient(
#     url=f"http://localhost:6333"
# )
qdrant_client = QdrantClient(
    url=config.QDRANT_URL,
    api_key=config.QDRANT_API_KEY  # For Qdrant Cloud only
)

In [4]:

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


async def ragas_faithfulness(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


async def ragas_response_relevancy(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)


async def ragas_context_precision(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_precision/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = LLMContextPrecisionWithoutReference(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_llm_based(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            reference=example.outputs["ground_truth"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = LLMContextRecall(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_non_llm(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/
    sample = SingleTurnSample(
            retrieved_contexts=run.outputs["retrieved_context"],
            reference_contexts=example.outputs["contexts"]
        )
    scorer = NonLLMContextRecall()

    return await scorer.single_turn_ascore(sample)


In [5]:
results = ls_client.evaluate(
    lambda x: rag_pipeline(x["question"], qdrant_client, session_id=0),
    data="rag-evaluation-dataset",
    evaluators=[
        ragas_faithfulness,
        ragas_response_relevancy,
        ragas_context_precision,
        ragas_context_recall_llm_based,
        ragas_context_recall_non_llm
    ],
    experiment_prefix="rag-evaluation-dataset"
)

# results = await ls_client.aevaluate(
#     target=lambda example: rag_pipeline(
#         example["question"],
#         qdrant_client,
#         session_id=0
#     ),
#     data="rag-evaluation-dataset",
#     evaluators=[
#         ragas_faithfulness,
#         ragas_response_relevancy,
#         ragas_context_precision,
#         ragas_context_recall_llm_based,
#         ragas_context_recall_non_llm,
#     ],
#     experiment_prefix="rag-evaluation-dataset"
# )

# results

# # Async main
# async def main():
#     results = await ls_client.aevaluate(
#         lambda x: rag_pipeline(
#             x["question"],
#             qdrant_client,
#             session_id=0
#         ),
#         data="rag-evaluation-dataset",
#         evaluators=[
#             ragas_faithfulness,
#             ragas_response_relevancy,
#             ragas_context_precision,
#             ragas_context_recall_llm_based,
#             ragas_context_recall_non_llm,
#         ],
#         experiment_prefix="rag-evaluation-dataset"
#     )
#     print(results)

# if __name__ == "__main__":
#     asyncio.run(main())

View the evaluation results for experiment: 'rag-evaluation-dataset-4184f6bb' at:
https://smith.langchain.com/o/02af5b7b-50d4-4c33-bcba-5ab8f98be641/datasets/e828eebe-7255-4ece-8e57-e4b1faa6de46/compare?selectedSessions=ddf8b3be-0321-4b18-998f-bf3035aa4610




0it [00:00, ?it/s]Error running target function: [Errno 2] No such file or directory: 'src/api/rag/prompts/rag_generation.yaml'
Traceback (most recent call last):
  File "/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1924, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/tmp/ipykernel_62463/708455139.py", line 2, in <lambda>
    lambda x: rag_pipeline(x["question"], qdrant_client, session_id=0),
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/philippe/Documents/Github/rag-demo/src/api/rag/retrieval.py", line 528, in rag_pipeline
    prompt = build_prompt(
             ^^^^^^^^^^^^^
  File "/home/philippe/Documents/Github/rag-demo/src/api/rag/retrieval.py", line 298, in build_prompt
    prompt_template = prompt_template_config(config.RAG_PROMPT_TEMPLATE_PATH, "rag_generation")
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [6]:
import langsmith
print(langsmith.__version__)

0.4.14
